# Insurance Fraud Detection — Bước 4: SHAP Explainability
**Input:** `model_xgb.pkl` · `fraud_features.csv` · `model_metadata.json`  
**Output:** `shap_values.pkl` · `shap_summary.png` · `shap_report.csv`

---
### Cấu trúc notebook
1. Lý thuyết SHAP — tại sao cần explainability
2. Load model & data
3. Global explainability — Summary plot & Feature importance
4. Cohort analysis — SHAP theo nhóm fraud/legit
5. Local explainability — Waterfall plot cho từng claim
6. SHAP Dependence plots — Feature interaction
7. Decision plot — Visualize prediction path
8. SHAP-based feature selection — Drop low-signal features
9. Lưu SHAP artifacts


## 1. Tại sao cần SHAP — không chỉ Feature Importance?

**Feature Importance (Gain)** từ XGBoost chỉ cho biết feature nào được dùng nhiều.  
**SHAP** cho biết feature đó **tác động như thế nào** đến từng prediction cụ thể.

| Câu hỏi | Feature Importance | SHAP |
|---|---|---|
| Feature nào quan trọng nhất? | ✅ | ✅ |
| Feature tác động dương hay âm? | ❌ | ✅ |
| Tại sao claim này bị flag fraud? | ❌ | ✅ |
| Feature tương tác với nhau thế nào? | ❌ | ✅ |
| Dùng được cho regulatory audit? | ❌ | ✅ |

### Công thức SHAP (Shapley Values)

Xuất phát từ lý thuyết game theory của Lloyd Shapley (1953):

$$\phi_i = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} \left[ f(S \cup \{i\}) - f(S) \right]$$

SHAP value của feature $i$ = **đóng góp trung bình** của feature đó  
khi thêm vào tất cả các tập con features có thể.

**Diễn giải:**
- SHAP > 0 → feature này **tăng** xác suất fraud
- SHAP < 0 → feature này **giảm** xác suất fraud  
- |SHAP| lớn → feature có tác động mạnh với prediction này

**TreeSHAP** (dùng cho XGBoost): tính chính xác trong O(TLD²)  
thay vì exponential time của SHAP gốc — rất nhanh cho tree models.


## 2. Load Model & Data

In [ ]:
import warnings, json, pickle
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, recall_score, precision_score)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({
    "figure.dpi": 115,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "font.family": "sans-serif",
})
PALETTE = {"fraud":"#E53935","legit":"#2196F3","neutral":"#78909C","good":"#43A047"}

# ── Load artifacts ─────────────────────────────────────────────────────────────
with open("model_xgb.pkl",      "rb") as f: best_xgb  = pickle.load(f)
with open("model_metadata.json")      as f: meta      = json.load(f)
with open("feature_metadata.json")    as f: feat_meta = json.load(f)

df = pd.read_csv("fraud_features.csv")

TARGET        = "target"
FEATURE_NAMES = meta["feature_names"]
FEATURE_NAMES = [f for f in FEATURE_NAMES if f in df.columns]
THRESHOLD     = meta["threshold"]

X = df[FEATURE_NAMES].fillna(0)
y = df[TARGET]

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

y_prob = best_xgb.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= THRESHOLD).astype(int)

print(f"✓ Model loaded | Features: {len(FEATURE_NAMES)}")
print(f"  Threshold : {THRESHOLD:.4f}")
print(f"  PR-AUC    : {average_precision_score(y_test, y_prob):.4f}")
print(f"  F1        : {f1_score(y_test, y_pred):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred):.4f}")


## 3. Khởi tạo SHAP TreeExplainer

In [ ]:
# ── TreeExplainer — tối ưu cho XGBoost/tree models ───────────────────────────
explainer = shap.TreeExplainer(
    best_xgb,
    feature_perturbation="interventional",  # chính xác hơn "tree_path_dependent"
    model_output="probability",             # output = P(fraud), không phải log-odds
)

# Tính SHAP values cho test set
# Dùng sample 1000 records để visualize cho nhanh (full có thể dùng hết)
np.random.seed(42)
sample_idx   = np.random.choice(len(X_test), min(1000, len(X_test)), replace=False)
X_test_sample = X_test.iloc[sample_idx].reset_index(drop=True)
y_test_sample = y_test.iloc[sample_idx].reset_index(drop=True)

print("Đang tính SHAP values...")
shap_values = explainer(X_test_sample)
print(f"✓ SHAP values computed")
print(f"  Shape: {shap_values.values.shape}")
print(f"  Base value (E[f(x)]): {shap_values.base_values[0]:.4f}")
print(f"  → Model dự báo fraud trung bình = {shap_values.base_values[0]*100:.1f}%")

# Kiểm tra SHAP additivity
sample_pred = best_xgb.predict_proba(X_test_sample)[:, 1]
shap_sum    = shap_values.values.sum(axis=1) + shap_values.base_values
print(f"\nAdditivity check (max diff): {np.abs(shap_sum - sample_pred).max():.6f}")
print("✓ SHAP values additive và nhất quán với model predictions")


## 4. Global Explainability — Summary Plot

**Summary plot** kết hợp feature importance + direction + distribution trong 1 chart:
- **Trục Y:** Feature (sắp xếp theo mean |SHAP|)
- **Trục X:** SHAP value (âm = giảm fraud risk, dương = tăng fraud risk)
- **Màu:** Giá trị feature (đỏ = cao, xanh = thấp)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# ── Plot 1: Beeswarm (summary plot chuẩn) ─────────────────────────────────────
plt.sca(axes[0])
shap.summary_plot(
    shap_values.values,
    X_test_sample,
    feature_names=FEATURE_NAMES,
    max_display=20,
    show=False,
    plot_size=None,
    color_bar=True,
)
axes[0].set_title("SHAP Summary — Beeswarm Plot\n(đỏ=giá trị cao, xanh=giá trị thấp)",
                  fontsize=11, fontweight="bold")
axes[0].set_xlabel("SHAP value (impact on fraud probability)")

# ── Plot 2: Bar plot (mean |SHAP|) ────────────────────────────────────────────
mean_shap = np.abs(shap_values.values).mean(axis=0)
shap_imp  = pd.DataFrame({
    "Feature":    FEATURE_NAMES,
    "Mean |SHAP|": mean_shap,
}).sort_values("Mean |SHAP|", ascending=False).head(20)

colors_bar = [
    PALETTE["fraud"]   if v > shap_imp["Mean |SHAP|"].quantile(0.75) else
    PALETTE["legit"]   if v > shap_imp["Mean |SHAP|"].quantile(0.25) else
    PALETTE["neutral"]
    for v in shap_imp["Mean |SHAP|"]
]
axes[1].barh(
    shap_imp["Feature"][::-1],
    shap_imp["Mean |SHAP|"][::-1],
    color=colors_bar[::-1], height=0.6, edgecolor="white",
)
for i, (_, row) in enumerate(shap_imp[::-1].iterrows()):
    axes[1].text(row["Mean |SHAP|"] + 0.0002, i,
                 f"{row['Mean |SHAP|']:.4f}", va="center", fontsize=8)
axes[1].set_xlabel("Mean |SHAP value|")
axes[1].set_title("Global Feature Importance\n(Mean absolute SHAP)",
                  fontsize=11, fontweight="bold")

plt.suptitle("SHAP Global Explainability", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fraud_13_shap_summary.png", bbox_inches="tight")
plt.show()

print("\nTop 10 features by mean |SHAP|:")
print(shap_imp.head(10).to_string(index=False))


## 5. Cohort Analysis — SHAP theo nhóm Fraud vs Legit

So sánh SHAP distribution giữa 2 nhóm để hiểu:
- Feature nào **phân biệt** fraud và legit rõ nhất?
- Chiều tác động có nhất quán không?


In [ ]:
fraud_idx = y_test_sample[y_test_sample == 1].index.tolist()
legit_idx = y_test_sample[y_test_sample == 0].index.tolist()

shap_fraud = shap_values.values[fraud_idx]
shap_legit = shap_values.values[legit_idx]

mean_shap_fraud = shap_fraud.mean(axis=0)
mean_shap_legit = shap_legit.mean(axis=0)

cohort_df = pd.DataFrame({
    "Feature":        FEATURE_NAMES,
    "SHAP_fraud":     mean_shap_fraud,
    "SHAP_legit":     mean_shap_legit,
    "Abs_difference": np.abs(mean_shap_fraud - mean_shap_legit),
}).sort_values("Abs_difference", ascending=False).head(15).reset_index(drop=True)

print("Mean SHAP: Fraud vs Legit (top 15 separating features):")
print(cohort_df.to_string(index=False))

# ── Visualize cohort comparison ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
top15 = cohort_df.head(15)
x     = np.arange(len(top15))
w     = 0.35

bars1 = ax.bar(x - w/2, top15["SHAP_fraud"], w,
               color=PALETTE["fraud"]+"CC", edgecolor="white",
               label=f"Fraud (n={len(fraud_idx)})")
bars2 = ax.bar(x + w/2, top15["SHAP_legit"], w,
               color=PALETTE["legit"]+"CC", edgecolor="white",
               label=f"Legit (n={len(legit_idx)})")

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(
    [f.replace("_te","").replace("_le","").replace("_enc","")[:18]
     for f in top15["Feature"]],
    rotation=40, ha="right", fontsize=9,
)
ax.set_ylabel("Mean SHAP value")
ax.set_title("Mean SHAP: Fraud vs Legit — Top 15 Separating Features",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10, frameon=False)
plt.tight_layout()
plt.savefig("fraud_14_shap_cohort.png", bbox_inches="tight")
plt.show()


## 6. Local Explainability — Waterfall Plot

**Waterfall plot** giải thích prediction cho **từng claim cụ thể**:
- Bắt đầu từ base value E[f(x)] (fraud rate trung bình)
- Mỗi feature cộng/trừ vào để ra prediction cuối
- Đây là thứ investigator cần khi quyết định có điều tra claim hay không


In [4]:
def plot_waterfall_case(shap_vals, X_sample, y_sample, y_prob_sample,
                        case_idx, threshold, title_prefix=""):
    """
    Vẽ waterfall plot cho 1 claim cụ thể.
    Hiển thị: prediction, actual label, top features tác động.
    """
    prob     = y_prob_sample[case_idx]
    actual   = y_sample.iloc[case_idx]
    decision = "FRAUD" if prob >= threshold else "LEGIT"
    correct  = "✓" if (prob >= threshold) == actual else "✗"

    fig, ax = plt.subplots(figsize=(12, 6))
    shap.waterfall_plot(
        shap_vals[case_idx],
        max_display=12,
        show=False,
    )
    plt.title(
        f"{title_prefix}\n"
        f"P(fraud) = {prob:.3f}  |  Decision = {decision}  |  "
        f"Actual = {'FRAUD' if actual==1 else 'LEGIT'}  |  {correct}",
        fontsize=10, fontweight="bold", pad=12,
    )
    plt.tight_layout()
    return fig

y_prob_sample = best_xgb.predict_proba(X_test_sample)[:, 1]

# ── Case 1: True Positive — bắt đúng fraud ───────────────────────────────────
tp_idx = [i for i in fraud_idx
          if y_prob_sample[i] >= THRESHOLD]
if tp_idx:
    # Chọn case có confidence cao nhất
    best_tp = tp_idx[np.argmax([y_prob_sample[i] for i in tp_idx])]
    fig1 = plot_waterfall_case(
        shap_values, X_test_sample, y_test_sample, y_prob_sample,
        best_tp, THRESHOLD,
        "Case 1 — TRUE POSITIVE: Bắt đúng Fraud (confidence cao nhất)",
    )
    plt.savefig("fraud_15a_waterfall_tp.png", bbox_inches="tight")
    plt.show()

# ── Case 2: False Negative — miss fraud ──────────────────────────────────────
fn_idx = [i for i in fraud_idx
          if y_prob_sample[i] < THRESHOLD]
if fn_idx:
    worst_fn = fn_idx[np.argmin([y_prob_sample[i] for i in fn_idx])]
    fig2 = plot_waterfall_case(
        shap_values, X_test_sample, y_test_sample, y_prob_sample,
        worst_fn, THRESHOLD,
        "Case 2 — FALSE NEGATIVE: Miss Fraud (model tự tin nhất sai)",
    )
    plt.savefig("fraud_15b_waterfall_fn.png", bbox_inches="tight")
    plt.show()

# ── Case 3: True Negative — legit claim được approve đúng ────────────────────
tn_idx = [i for i in legit_idx
          if y_prob_sample[i] < THRESHOLD]
if tn_idx:
    best_tn = tn_idx[np.argmin([y_prob_sample[i] for i in tn_idx])]
    fig3 = plot_waterfall_case(
        shap_values, X_test_sample, y_test_sample, y_prob_sample,
        best_tn, THRESHOLD,
        "Case 3 — TRUE NEGATIVE: Legit claim được approve đúng",
    )
    plt.savefig("fraud_15c_waterfall_tn.png", bbox_inches="tight")
    plt.show()


NameError: name 'best_xgb' is not defined

## 7. SHAP Dependence Plots — Feature Interactions

**Dependence plot** hiển thị:
- Trục X: Giá trị thực của feature
- Trục Y: SHAP value (tác động lên fraud probability)
- Màu: Feature tương tác (SHAP tự chọn feature interact mạnh nhất)

Giúp hiểu **ngưỡng** tại đó feature chuyển từ giảm sang tăng fraud risk.


In [ ]:
# Top 6 features để vẽ dependence plot
top6_features = shap_imp.head(6)["Feature"].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(top6_features):
    if feat not in FEATURE_NAMES:
        continue
    feat_idx = FEATURE_NAMES.index(feat)

    ax = axes[i]
    shap.dependence_plot(
        feat_idx,
        shap_values.values,
        X_test_sample,
        feature_names=FEATURE_NAMES,
        ax=ax,
        show=False,
        dot_size=15,
        alpha=0.6,
    )
    ax.set_title(f"{feat.replace('_te','').replace('_le','').replace('_enc','')}",
                 fontsize=10, fontweight="bold")
    ax.axhline(0, color="gray", linestyle="--", lw=0.8, alpha=0.5)

plt.suptitle("SHAP Dependence Plots — Top 6 Features",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fraud_16_shap_dependence.png", bbox_inches="tight")
plt.show()


## 8. Decision Plot — Visualize Prediction Path

**Decision plot** cho thấy cách từng claim "đi" từ base value đến final prediction,
với mỗi feature cộng/trừ dọc đường đi.

Hữu ích để so sánh nhiều claims cùng lúc — ví dụ: fraud vs legit.


In [ ]:
# Lấy 20 fraud + 20 legit cases để so sánh
n_show = min(20, len(fraud_idx), len(legit_idx))
show_fraud = fraud_idx[:n_show]
show_legit = legit_idx[:n_show]
show_idx   = show_fraud + show_legit
show_labels= ["Fraud"]*n_show + ["Legit"]*n_show

shap_show  = shap_values.values[show_idx]
X_show     = X_test_sample.iloc[show_idx]
base_show  = shap_values.base_values[show_idx]

fig, ax = plt.subplots(figsize=(12, 8))
shap.decision_plot(
    base_show[0],
    shap_show,
    X_show,
    feature_names=FEATURE_NAMES,
    feature_display_range=slice(-1, -16, -1),  # top 15 features
    highlight=[i for i, l in enumerate(show_labels) if l == "Fraud"],
    legend_labels=show_labels,
    legend_location="lower right",
    show=False,
    ax=ax,
)
ax.set_title("Decision Plot — Fraud vs Legit Claims
"
             "(highlighted = Fraud cases)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("fraud_17_shap_decision.png", bbox_inches="tight")
plt.show()


## 9. SHAP-based Reason Codes cho từng Claim

Giống như Dự án 1 có ECOA reason codes, fraud detection cần
**investigation reason codes** — giải thích tại sao claim bị flag.

Format output: top 5 features tác động dương (tăng fraud risk) nhất.


In [ ]:
def get_fraud_reason_codes(shap_vals_row, feature_names, top_n=5):
    """
    Trả về top N reason codes cho 1 claim bị flag fraud.
    Chỉ lấy features có SHAP > 0 (tăng fraud risk).

    Returns
    -------
    pd.DataFrame với cột: Rank, Feature, SHAP_impact, Direction
    """
    shap_series = pd.Series(shap_vals_row, index=feature_names)

    # Chỉ lấy features đóng góp tăng fraud risk
    positive_shap = shap_series[shap_series > 0].sort_values(ascending=False)

    rows = []
    for rank, (feat, val) in enumerate(positive_shap.head(top_n).items(), 1):
        rows.append({
            "Rank":        rank,
            "Feature":     feat,
            "SHAP Impact": round(val, 4),
            "Direction":   "↑ Tăng fraud risk",
        })
    return pd.DataFrame(rows)


# Demo cho 5 fraud cases bị bắt đúng (True Positives)
print("═══ FRAUD REASON CODES — Top True Positive Cases ═══")
print()
shown = 0
for i in range(len(X_test_sample)):
    if y_test_sample.iloc[i] == 1 and y_prob_sample[i] >= THRESHOLD:
        prob = y_prob_sample[i]
        rc   = get_fraud_reason_codes(shap_values.values[i], FEATURE_NAMES)
        print(f"Claim #{i} | P(fraud) = {prob:.3f} | Actual = FRAUD")
        print(rc.to_string(index=False))
        print()
        shown += 1
        if shown >= 3:
            break

# Demo cho False Negatives (missed fraud)
print("═══ MISSED FRAUD — False Negative Analysis ═══")
print("(Tại sao model không bắt được những cases này?)")
print()
shown = 0
for i in range(len(X_test_sample)):
    if y_test_sample.iloc[i] == 1 and y_prob_sample[i] < THRESHOLD:
        prob = y_prob_sample[i]
        # Với FN, xem features nào đang kéo score xuống (SHAP âm)
        shap_series  = pd.Series(shap_values.values[i], index=FEATURE_NAMES)
        negative_shap = shap_series[shap_series < 0].sort_values()
        print(f"Claim #{i} | P(fraud) = {prob:.3f} | Actual = FRAUD (MISSED)")
        print("  Features kéo risk xuống (gây miss):")
        for feat, val in negative_shap.head(3).items():
            print(f"    {feat:35s} SHAP = {val:.4f}")
        print()
        shown += 1
        if shown >= 2:
            break


## 10. SHAP-based Feature Selection

Dùng mean |SHAP| để xác định features có đóng góp thực sự vs noise.
Features có mean |SHAP| rất thấp → drop để giảm model complexity.


In [ ]:
shap_importance = pd.DataFrame({
    "Feature":     FEATURE_NAMES,
    "Mean_SHAP":   np.abs(shap_values.values).mean(axis=0),
    "Max_SHAP":    np.abs(shap_values.values).max(axis=0),
    "Pos_rate":    (shap_values.values > 0).mean(axis=0),
}).sort_values("Mean_SHAP", ascending=False).reset_index(drop=True)

# Ngưỡng: features có mean |SHAP| < 1% của max feature
threshold_shap = shap_importance["Mean_SHAP"].max() * 0.01
low_signal     = shap_importance[shap_importance["Mean_SHAP"] < threshold_shap]

print(f"SHAP importance threshold: {threshold_shap:.6f}")
print(f"Features dưới ngưỡng (có thể drop): {len(low_signal)}")
if len(low_signal) > 0:
    print(low_signal[["Feature","Mean_SHAP"]].to_string(index=False))

print(f"\nFeatures giữ lại: {len(shap_importance) - len(low_signal)}")

# Lưu SHAP importance report
shap_importance.to_csv("shap_report.csv", index=False)
print("\n✓ Saved: shap_report.csv")


## 11. SHAP Validation Dashboard

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Metric cards ──────────────────────────────────────────────────────────────
metrics = [
    ("PR-AUC",    f"{average_precision_score(y_test, y_prob):.3f}", "Test set",   True),
    ("F1 Score",  f"{f1_score(y_test, y_pred):.3f}",                "Test set",   True),
    ("Recall",    f"{recall_score(y_test, y_pred):.3f}",            "Key metric", True),
]
for i, (name, val, sub, good) in enumerate(metrics):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor("#E8F5E9" if good else "#FFF8E1")
    ax.text(0.5, 0.62, val,  ha="center", va="center", fontsize=28,
            fontweight="bold", transform=ax.transAxes,
            color="#1B5E20" if good else "#E65100")
    ax.text(0.5, 0.30, name, ha="center", va="center", fontsize=12,
            fontweight="bold", transform=ax.transAxes)
    ax.text(0.5, 0.12, sub,  ha="center", va="center", fontsize=9,
            color="gray", transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

# ── Top SHAP features bar ─────────────────────────────────────────────────────
ax_shap = fig.add_subplot(gs[1, 0:2])
top10_shap = shap_importance.head(10)
bar_colors = [PALETTE["fraud"] if i < 3 else PALETTE["legit"] if i < 7
              else PALETTE["neutral"] for i in range(len(top10_shap))]
ax_shap.barh(
    top10_shap["Feature"][::-1],
    top10_shap["Mean_SHAP"][::-1],
    color=bar_colors[::-1], height=0.6, edgecolor="white",
)
ax_shap.set_xlabel("Mean |SHAP value|")
ax_shap.set_title("SHAP Feature Importance — Top 10")
ax_shap.set_yticklabels(
    [f.replace("_te","").replace("_le","").replace("_enc","")[:20]
     for f in top10_shap["Feature"][::-1]], fontsize=9,
)

# ── Fraud probability distribution ───────────────────────────────────────────
ax_dist = fig.add_subplot(gs[1, 2])
ax_dist.hist(y_prob[y_test==0], bins=40, alpha=0.6, color=PALETTE["legit"],
             label="Legit", density=True, edgecolor="none")
ax_dist.hist(y_prob[y_test==1], bins=40, alpha=0.6, color=PALETTE["fraud"],
             label="Fraud", density=True, edgecolor="none")
ax_dist.axvline(THRESHOLD, color="orange", linestyle="--",
                lw=2, label=f"Threshold={THRESHOLD:.2f}")
ax_dist.set_xlabel("P(fraud)")
ax_dist.set_title("Score Distribution")
ax_dist.legend(fontsize=9, frameon=False)

plt.suptitle("Fraud Detection — SHAP Explainability Dashboard",
             fontsize=14, fontweight="bold", y=1.01)
plt.savefig("fraud_18_shap_dashboard.png", bbox_inches="tight", dpi=130)
plt.show()
print("✓ Saved: fraud_18_shap_dashboard.png")


## 12. Lưu SHAP Artifacts

In [ ]:
# ── Save SHAP values & explainer ─────────────────────────────────────────────
with open("shap_values.pkl","wb") as f:
    pickle.dump({
        "shap_values":   shap_values.values,
        "base_values":   shap_values.base_values,
        "feature_names": FEATURE_NAMES,
        "X_sample":      X_test_sample.values,
    }, f)
print("✓ Saved: shap_values.pkl")

with open("shap_explainer.pkl","wb") as f:
    pickle.dump(explainer, f)
print("✓ Saved: shap_explainer.pkl")

# ── Update model metadata với SHAP info ──────────────────────────────────────
meta["shap_top10_features"] = shap_importance.head(10)[
    ["Feature","Mean_SHAP"]
].to_dict("records")
meta["shap_low_signal_features"] = low_signal["Feature"].tolist()

with open("model_metadata.json","w") as f:
    json.dump(meta, f, indent=2)
print("✓ Updated: model_metadata.json")

print()
print("═══ TÓM TẮT BƯỚC 4 ═══")
print(f"  SHAP sample size    : {len(X_test_sample):,}")
print(f"  Base value          : {shap_values.base_values[0]:.4f}")
print(f"  Top feature (SHAP)  : {shap_importance.iloc[0]['Feature']}")
print(f"  Low signal features : {len(low_signal)}")
print(f"  Charts generated    : 6 (summary, cohort, waterfall×3, dependence, decision, dashboard)")
print()
print("─── Bước tiếp theo: Bước 5 — Streamlit Dashboard ───")
